note: implemented from ai-engineering-from-scratch exactly. personal comments added to help understanding.

# gini impurity and entropy

In [1]:
import math

# gini impurity defined as Gini(S) = 1 - sum(p_k^2)

def gini_impurity(labels):
    n = len(labels)
    if n == 0:
        return 0.0 # if no labels exist, return 0
    counts = {}
    for label in labels: # add one to the label counts for each occurrence
        counts[label] = counts.get(label, 0) + 1

    return 1.0 - sum((c/n) ** 2 for c in counts.values()) # implements the formula


# entropy defined as Entropy(S) = -sum(p_k * log2(p_k))

def entropy(labels):
    # following same basic logic as gini
    n = len(labels)
    if n == 0:
        return 0.0
    counts = {}
    for label in labels:
        counts[label]= counts.get(label, 0)+ 1

    # implement the formula
    # only if c > 0 to guarantee that log2 doesn't throw an error
    return -sum((c/n)* math.log2(c/n) for c in counts.values() if c>0)

# finding best splits
goal: try every feature at all thresholds and return one with highest info gain.

In [ ]:
# info gain defined as IG(S, feature, threshold) = Impurity(S) - weighted_avg(Impurity(S_left), Impurity(S_right))

def information_gain(parent_labels, left_labels, right_labels, criterion="gini"):
    measure = gini_impurity if criterion == "gini" else entropy # chooses measure of impurity

    # number of labels on each node
    n = len(parent_labels)
    n_left = len(left_labels)
    n_right = len(right_labels)

    # if no samples went to one or both children, no information was gained
    if n_left == 0 or n_right == 0:
        return 0.0

    # measure impurities of parent and children
    parent_impurity = measure(parent_labels)
    child_impurity = (
        (n_left / n) * measure(left_labels) +
        (n_right / n) * measure(right_labels)
    )
    # implement the formula
    return parent_impurity - child_impurity


# decision tree class
features: recursive splitting, prediction and feature importance tracking.

`_build` makes the tree and stops if a node is pure or hits a pre-pruning limit; otherwise takes best split and recurses into the children.

In [ ]:
import random

class DecisionTree:
    def __init__(self, max_depth=None, min_samples_split=2,
                 min_samples_leaf=1, criterion="gini",
                 max_features=None):
        self.max_depth = max_depth # depth of tree
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.criterion = criterion # impurity measure
        self.max_features = max_features
        self.tree = None # the tree itself
        self.feature_importances_ = None # holds feature importance list

    def fit(self, X, y):
        self.n_features = len(X[0])

        # initialize feature importances with 0's
        self.feature_importances_ = [0.0] * self.n_features
        self.n_samples = len(X)
        self.tree = self._build(X, y, depth=0)

        total = sum(self.feature_importances_)

        # measure feature importance by dividing by sum of importance scores
        if total > 0:
            self.feature_importances_ = [
                fi / total for fi in self.feature_importances_
            ]

    def predict(self, X):
        # predicts for multiple samples
        return [self._predict_one(x, self.tree) for x in X]

    def _build(self, X, y, depth):
        # makes a node containing only one unique value a leaf node
        if len(set(y)) == 1:
            return {"leaf": True, "value": y[0]}

        # makes leaf node if the max depth has been exceeded
        if self.max_depth is not None and depth >= self.max_depth:
            return self._make_leaf(y)

        # make a leaf if the current node goes under min samples
        # prevents nodes from splitting if we are under the minimum amount currently
        if len(y) < self.min_samples_split:
            return self._make_leaf(y)

        # finding splits
        best_feature, best_threshold, best_gain = self._best_split(X, y)

        # if no information gains have been made, make it a leaf
        if best_feature is None or best_gain <= 0:
            return self._make_leaf(y)

        left_X, left_y, right_X, right_y = self._split_data(
            X, y, best_feature, best_threshold
        )

        # if there are too few samples, make it a leaf node instead of splitting further
        if len(left_y) < self.min_samples_leaf or len(right_y) < self.min_samples_leaf:
            return self._make_leaf(y)

        # MDI calculation
        weight = len(y) / self.n_samples
        self.feature_importances_[best_feature] += weight * best_gain

        # recursive behavior:
        return {
            "leaf": False,
            "feature": best_feature,
            "threshold": best_threshold,
            "left": self._build(left_X, left_y, depth + 1),
            "right": self._build(right_X, right_y, depth + 1),
        }

    def _make_leaf(self, y):
        # counting mechanism
        counts = {}
        for label in y:
            counts[label] = counts.get(label, 0) + 1
        # returns majority class as this leaf's prediction
        # leaf value comes from the labels of y (which is a subset of the full label list, given by the path the decision tree takes)
        return {"leaf": True, "value": max(counts, key=counts.get)}

    def _best_split(self, X, y):
        # if nothing is found, return these
        # fits in with _build since it will return a leaf node if no info is gained
        best_feature = None
        best_threshold = None
        best_gain = -1.0

        # if-else controls how many features to consider; ready for random forests
        if self.max_features == "sqrt":
            k = max(1, int(math.sqrt(self.n_features)))
            feature_indices = random.sample(range(self.n_features), k)
        elif isinstance(self.max_features, int):
            if self.max_features < 1:
                raise ValueError("max_features must be at least 1 when given as an integer")
            k = min(self.max_features, self.n_features)
            feature_indices = random.sample(range(self.n_features), k)
        else:
            feature_indices = list(range(self.n_features))

        for feature_idx in feature_indices:
            # sort the unique feature values
            values = sorted(set(X[i][feature_idx] for i in range(len(X))))
            # if 1 or less unique values of a current feature, skip
            if len(values) <= 1:
                continue

            for i in range(len(values) - 1):
                # trying the midpoints
                threshold = (values[i] + values[i + 1]) / 2.0
                # splits the y data according to their feature value
                left_y = [y[j] for j in range(len(X)) if X[j][feature_idx] <= threshold]
                right_y = [y[j] for j in range(len(X)) if X[j][feature_idx] > threshold]

                # if we have less samples in the children than minimum, skip
                if len(left_y) < self.min_samples_leaf or len(right_y) < self.min_samples_leaf:
                    continue

                # calculate gains
                gain = information_gain(y, left_y, right_y, self.criterion)
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature_idx
                    best_threshold = threshold

        return best_feature, best_threshold, best_gain

    def _split_data(self, X, y, feature, threshold):
        # splits the data into the left and right children once the best split is found
        left_X, left_y, right_X, right_y = [], [], [], []
        for i in range(len(X)):
            if X[i][feature] <= threshold:
                left_X.append(X[i])
                left_y.append(y[i])
            else:
                right_X.append(X[i])
                right_y.append(y[i])
        return left_X, left_y, right_X, right_y

    def _predict_one(self, x, node):
        # recursively goes down tree according to threshold rules until it hits a leaf, returns leaf value
        if node["leaf"]:
            return node["value"]
        if x[node["feature"]] <= node["threshold"]:
            return self._predict_one(x, node["left"])
        return self._predict_one(x, node["right"])

# random forest

In [ ]:
class RandomForest:
    def __init__(self, n_trees=100, max_depth=None,
                 min_samples_split=2, max_features="sqrt",
                 criterion="gini"):
        self.n_trees = n_trees # no. of trees
        self.max_depth = max_depth # max depth of trees
        self.min_samples_split = min_samples_split # min amt of samples in a node to consider splitting
        self.max_features = max_features # max features considered in the tree
        self.criterion = criterion # impurity measure
        self.trees = [] # list of trees

    def fit(self, X, y):
        n = len(X)
        for _ in range(self.n_trees):
            # randomly sampling data by bootstrap
            indices = [random.randint(0, n - 1) for _ in range(n)]
            X_boot = [X[i] for i in indices]
            y_boot = [y[i] for i in indices]

            # initiate
            tree = DecisionTree(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=self.max_features,
                criterion=self.criterion,
            )
            # use tree fitting function and append tree
            tree.fit(X_boot, y_boot)
            self.trees.append(tree)

    def predict(self, X):
        # use tree prediction method for all trees
        all_preds = [tree.predict(X) for tree in self.trees]
        predictions = []

        # gather votes for each class and choose the maximum out of the trees
        for i in range(len(X)): # go down list of all samples
            votes = {}
            for preds in all_preds:
                v = preds[i]
                votes[v] = votes.get(v, 0) + 1
            predictions.append(max(votes, key=votes.get))
        return predictions